# CS604 Colab launcher — Ana: CodaBench submission with kNN re-ranking (no training)

Re-ranks the model's top-5 guesses with each candidate's nearest training photos (`scripts/predict_rerank.py`): `score = log p + beta * knn`, knn = mean of the 3 best cosines to the candidate's training photos. Settings K = 5, m = 3, beta = 20 were fixed on the all-class holdout (both held-out halves chose them independently; +1.29 points there) and are **not** tuned on CodaBench.

Checkpoint: your full-scale **LoRA r24 without extras** (`lora_20261020_062136`, CodaBench 0.784 plain), the recipe the settings were chosen on. The bank is all 450,000 training photos, embedded once by this checkpoint and cached on Drive for the test phase.

Writes, in `CS604_RUNS/rerank_submission/`:
- `val_team_08.zip`: **the re-ranked submission**;
- `val_team_08_plain.zip`: the same pass without re-ranking (should reproduce 0.784; submit only if you want the paired check);
- `rerank_stats.json`: how many predictions changed, extra time per photo.

About 35 min on an A100: data ~15 min, bank ~15 min (once), predictions a few minutes. When the test split is released, set `SPLITS = ["val", "test"]` and Run all again: the bank is reused. Use **Runtime → Run all**.


In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "ana-branch"
RUN_NAME = "lora_20261020_062136"  # full-scale LoRA r24 without extras (CodaBench 0.784)
TEAM = "08"
SPLITS = ["val"]  # add "test" when the test split is released
K, M, BETA = 5, 3, 20.0  # fixed on the all-class holdout; do not tune on CodaBench
OUT_NAME = "rerank_submission"
# End the runtime after the last cell, or as soon as training fails.
AUTO_DISCONNECT = True


def end_runtime(reason):
    """Flush Drive writes, then release the GPU so it stops using compute units."""
    print(reason, flush=True)
    if not AUTO_DISCONNECT:
        return
    from google.colab import drive, runtime
    try:
        drive.flush_and_unmount()  # without this the last checkpoint can be lost
        print("Drive flushed and unmounted", flush=True)
    except Exception as error:
        print(f"Drive flush failed: {error}", flush=True)
    runtime.unassign()

## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

## 4. Download data to `/content/data`

In [ ]:
!python scripts/download_data.py --dest {DATA_DIR}

## 5. Predict with re-ranking

In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

RUN_DIR = Path(DRIVE_OUT) / RUN_NAME
assert (RUN_DIR / "last.pt").is_file(), f"no checkpoint in {RUN_DIR}"
OUT = Path(DRIVE_OUT) / OUT_NAME
BANK = OUT / f"bank_{RUN_NAME}.pt"
for split in SPLITS:
    if split == "test" and not ((Path(DATA_DIR) / "test.txt").is_file() or (Path(DATA_DIR) / "test").is_dir()):
        print("test split not available yet; skipping"); continue
    proc = subprocess.run([sys.executable, "-u", "scripts/predict_rerank.py", "--run_dir", str(RUN_DIR),
                           "--data_root", DATA_DIR, "--split", split, "--team", TEAM, "--out_dir", str(OUT / split),
                           "--bank", str(BANK), "--K", str(K), "--m", str(M), "--betas", str(BETA)])
    if proc.returncode != 0:
        end_runtime(f"prediction failed (exit {proc.returncode})")
        raise SystemExit("prediction failed")
    for name in (f"{split}_team_{TEAM}", f"{split}_team_{TEAM}_plain"):
        folder = OUT / split / name
        folder.mkdir(exist_ok=True)
        shutil.copy(OUT / split / f"{name}.txt", folder / f"{split}_team_{TEAM}.txt")  # CodaBench expects this file name
        shutil.make_archive(str(OUT / split / name), "zip", root_dir=folder, base_dir=f"{split}_team_{TEAM}.txt")
        print("zip:", OUT / split / f"{name}.zip")
    print(json.dumps(json.loads((OUT / split / "rerank_stats.json").read_text()), indent=2))


## 6. Submit

Submit `rerank_submission/val/val_team_08.zip` (re-ranked) to CodaBench. Inside each zip the file is named `val_team_08.txt`, as CodaBench expects. Compare with the plain 0.784 of this checkpoint; the all-class holdout predicts roughly +1.3.

## 7. End the runtime

In [ ]:
end_runtime("all cells finished; ending the runtime")